# Exercise 3: JOINs and Relationships

**Learning Objectives:**
- Understand table structures with DESCRIBE
- INNER JOIN between two tables
- Multiple JOINs across three tables
- Aggregations with GROUP BY after JOINs

**Estimated Time:** 30-40 minutes

**Data Model:**
```
products ──────────────────┐
   ProductSubcategoryID ───┼──► product_subcategories
                           │      ProductCategoryID ───► product_categories
```

In [ ]:
import duckdb
conn = duckdb.connect(':memory:')

## Setup: Load Data

Run this cell to load the required tables:

In [ ]:
# Load product tables
conn.execute("""
    CREATE TABLE products AS 
    SELECT * FROM read_csv_auto('../sample_data/AW_CSV/Production.Product.csv')
""")

conn.execute("""
    CREATE TABLE product_categories AS 
    SELECT * FROM read_csv_auto('../sample_data/AW_CSV/Production.ProductCategory.csv')
""")

conn.execute("""
    CREATE TABLE product_subcategories AS 
    SELECT * FROM read_csv_auto('../sample_data/AW_CSV/Production.ProductSubcategory.csv')
""")

print("✓ Tables loaded: products, product_categories, product_subcategories")

---
## Task 1: Understand Table Structures

Before we can JOIN tables, we need to find the **key columns**.

### 1a) Structure of `products`

Show the columns of the `products` table using `DESCRIBE`.

**Look for:** Which column connects to subcategories? (Hint: contains 'Subcategory')

In [ ]:
# Your code here:


### 1b) Structure of `product_subcategories`

Show the columns of `product_subcategories`.

**Look for:** 
- Primary Key (ID of the subcategory)
- Foreign Key to the category table

In [ ]:
# Your code here:


### 1c) Structure of `product_categories`

Show the columns of `product_categories`.

**Look for:** Primary Key and Name column

In [ ]:
# Your code here:


---
## Task 2: Simple JOIN (2 Tables)

Join `products` with `product_subcategories` and show:
- `Name` from products (as `product_name`)
- `Name` from product_subcategories (as `subcategory_name`)
- `ListPrice` from products

**Show only the first 10 results!**

**JOIN Condition:** 
```
products.ProductSubcategoryID = product_subcategories.ProductSubcategoryID
```

**Syntax:**
```sql
SELECT 
    p.Name AS product_name,
    s.Name AS subcategory_name,
    p.ListPrice
FROM products p
INNER JOIN product_subcategories s 
    ON p.ProductSubcategoryID = s.ProductSubcategoryID
LIMIT 10
```

In [ ]:
# Your code here:


---
## Task 3: Multiple JOINs (3 Tables)

Extend the query and join **all three tables**:

Show:
- Product name
- Subcategory name
- **Category name** (NEW!)
- ListPrice

**Sort by:** Category name, then by ListPrice descending

**JOIN Conditions:**
1. `products.ProductSubcategoryID = product_subcategories.ProductSubcategoryID`
2. `product_subcategories.ProductCategoryID = product_categories.ProductCategoryID`

**Syntax:**
```sql
SELECT ...
FROM products p
INNER JOIN product_subcategories s ON p.ProductSubcategoryID = s.ProductSubcategoryID
INNER JOIN product_categories c ON s.ProductCategoryID = c.ProductCategoryID
ORDER BY c.Name, p.ListPrice DESC
```

In [ ]:
# Your code here:


---
## Task 4: Aggregation with JOINs

### 4a) Product count per category

Count how many products exist **per category**.

**Expected Columns:** category_name, product_count

**Sort by:** Count descending

**Expected Result:** 4 categories (Accessories, Bikes, Clothing, Components)

In [ ]:
# Your code here:


### 4b) Average price per category

Calculate the **average ListPrice** per category.

**Expected Columns:** category_name, avg_price (rounded to 2 decimal places)

**Tip:** Use `ROUND(AVG(p.ListPrice), 2)`

In [ ]:
# Your code here:


### 4c) Most expensive product per category

Find the **most expensive product** in each category.

**Expected Columns:** category_name, product_name, max_price

**Tip:** You need a subquery or Window Function

**Simple approach with subquery:**
```sql
SELECT c.Name, p.Name, p.ListPrice
FROM products p
JOIN ... ON ...
JOIN ... ON ...
WHERE p.ListPrice = (
    SELECT MAX(p2.ListPrice) 
    FROM products p2 
    JOIN product_subcategories s2 ON p2.ProductSubcategoryID = s2.ProductSubcategoryID
    WHERE s2.ProductCategoryID = c.ProductCategoryID
)
```

In [ ]:
# Your code here:


---
## Task 5: Sales Data Analysis

Now let's load additional tables for sales analysis:

In [ ]:
# Load sales tables
conn.execute("""
    CREATE TABLE sales_orders AS 
    SELECT * FROM read_csv_auto('../sample_data/AW_CSV/Sales.SalesOrderHeader.csv')
""")

conn.execute("""
    CREATE TABLE sales_details AS 
    SELECT * FROM read_csv_auto('../sample_data/AW_CSV/Sales.SalesOrderDetail.csv')
""")

conn.execute("""
    CREATE TABLE territories AS 
    SELECT * FROM read_csv_auto('../sample_data/AW_CSV/Sales.SalesTerritory.csv')
""")

print("✓ Sales tables loaded: sales_orders, sales_details, territories")

### 5a) Top 10 products by sales quantity

Show the **10 best-selling products** (by `OrderQty`).

**Tables:** `sales_details` JOIN `products`
**JOIN Column:** `ProductID`
**Aggregation:** `SUM(OrderQty)`

**Expected Columns:** product_name, total_quantity
**Sort by:** total_quantity descending, LIMIT 10

In [ ]:
# Your code here:


### 5b) Revenue per category

Which category has the **highest total revenue** (`LineTotal`)?

**Table Chain:**
```
sales_details ─► products ─► product_subcategories ─► product_categories
```

**Expected Columns:** category_name, total_revenue (formatted with ROUND)
**Sort by:** total_revenue descending

In [ ]:
# Your code here:


---
## Bonus Task ⭐⭐

Create a **comprehensive territory analysis** with:

| Column | Description |
|--------|-------------|
| territory_name | Name of the territory |
| order_count | Number of orders |
| total_revenue | Total revenue (TotalDue) |
| avg_order_value | Average order value |

**Tables:** `sales_orders` JOIN `territories`
**JOIN Column:** `TerritoryID`

**Sort by:** total_revenue descending

In [ ]:
# Your code here:


---
## 🎉 Congratulations!

You now master JOINs in DuckDB!

**What you learned:**
- ✅ Analyze table structures with DESCRIBE
- ✅ INNER JOIN between two tables
- ✅ Multiple JOINs across three+ tables
- ✅ Aggregations after JOINs (COUNT, AVG, SUM)
- ✅ Complex analyses with multiple tables